[Énoncés : Examen blanc C — Problèmes moins prévisibles](../examen_blanc_C.ipynb)

# Corrigé — Examen blanc C — Problèmes moins prévisibles

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec tous les tests (y compris les cas cachés de la correction). Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 59 — Premier mot suffisamment long

**Idée** : `phrase.split()` **sans argument** découpe sur n'importe quelle suite d'espaces et ignore ceux du début et de la fin. Avec `split(" ")`, on obtiendrait des mots vides. On renvoie le premier mot assez long, mis en minuscules.

**Complexité** : O(n).

In [ ]:
def premier_mot_long(phrase: str, minimum: int) -> str:
    for mot in phrase.split():          # split() sans argument gère les espaces multiples et aux bords
        if len(mot) >= minimum:
            return mot.lower()
    return ""


verifier(premier_mot_long, [
    (('un joli paysage', 5), 'paysage'),
    (('salut toi', 6), ''),
    (('', 1), ''),   # phrase vide
    (('  Bonjour   le monde ', 3), 'bonjour'),   # espaces multiples et majuscule
    (('Il FAIT beau', 4), 'fait'),
    (('abc', 3), 'abc'),   # exactement le minimum
])

## Exercice 60 — Case la plus proche d'une valeur cible

**Idée** : parcourir toutes les cases ligne par ligne, de gauche à droite, en gardant la meilleure. Grâce à l'ordre de parcours, une comparaison **stricte** garde automatiquement la plus haute puis la plus à gauche en cas d'égalité.

**Complexité** : O(lignes × colonnes).

In [ ]:
def case_plus_proche(grille: list[list[int]], cible: int) -> list[int]:
    meilleur = None
    for i, ligne in enumerate(grille):
        for j, valeur in enumerate(ligne):
            ecart = abs(valeur - cible)
            if meilleur is None or ecart < meilleur[0]:   # < strict : on garde la première trouvée
                meilleur = (ecart, i, j)
    return [meilleur[1], meilleur[2]]


verifier(case_plus_proche, [
    (([[8, 2], [7, 5]], 6), [1, 0]),
    (([[5]], 100), [0, 0]),
    (([[1, 2, 3]], 2), [0, 1]),
    (([[10, 20], [30, 40]], 25), [0, 1]),   # égalité : la plus haute
    (([[-5, 0], [5, 9]], -4), [0, 0]),
    (([[4, 4], [4, 4]], 4), [0, 0]),
])

## Exercice 61 — Premier découpage où la gauche domine

**Idée** : calculer le total une fois, puis faire avancer la somme de gauche. La somme de droite s'obtient par soustraction (`total - gauche`), sans boucle. C'est le principe des sommes préfixes (thème 6).

**Complexité** : O(n). Recalculer `sum(nombres[:i+1])` et `sum(nombres[i+1:])` à chaque indice coûterait O(n²).

In [ ]:
def premier_decoupage_dominant(nombres: list[int]) -> int:
    total = sum(nombres)
    gauche = 0
    for i in range(len(nombres) - 1):        # la partie droite doit rester non vide
        gauche += nombres[i]
        if gauche > total - gauche:          # droite = total − gauche, sans la recalculer
            return i
    return -1


verifier(premier_decoupage_dominant, [
    (([3, 1, 2],), 1),
    (([5],), -1),
    (([],), -1),
    (([1, 1],), -1),   # égalité : pas strictement supérieur
    (([2, 1],), 0),
    (([1, 2, 3, 10],), -1),
    (([-1, -2, -3],), 0),   # valeurs négatives
    (([0, 0, 1, 0],), 2),
])

In [ ]:
# Test de vitesse
chrono(premier_decoupage_dominant, [1] * 29_999 + [30_000], attendu=-1)
chrono(premier_decoupage_dominant, list(range(30_000)), attendu=21213)
chrono(premier_decoupage_dominant, [2] * 29_999 + [60_000], attendu=-1)

## Exercice 62 — Nombre maximal de personnes en réunion

**Idée** : transformer chaque plage en deux **événements** (arrivée +1, départ −1), les trier par instant, puis les parcourir en tenant le compte des présents. Le piège de l'énoncé (« part à t, arrive à t ») se règle par l'ordre de tri : à instant égal, le départ `(t, -1)` est trié avant l'arrivée `(t, 1)`.

**Complexité** : O(n log n).

In [ ]:
def occupation_max(plages: list[list[int]]) -> int:
    evenements = []
    for debut, fin in plages:
        evenements.append((debut, 1))      # une arrivée
        evenements.append((fin, -1))       # un départ
    # trié par instant ; à instant égal, (t, -1) passe avant (t, 1) : on fait partir avant de faire entrer
    evenements.sort()
    presents = maximum = 0
    for _, changement in evenements:
        presents += changement
        maximum = max(maximum, presents)
    return maximum


verifier(occupation_max, [
    (([[9, 11], [10, 12], [11, 13]],), 2),
    (([[1, 3], [1, 3], [2, 4]],), 3),
    (([],), 0),
    (([[1, 2]],), 1),
    (([[1, 2], [2, 3], [3, 4]],), 1),   # chacun part quand le suivant arrive
    (([[1, 10], [2, 3], [4, 5], [6, 7]],), 2),
    (([[0, 5], [1, 5], [2, 5], [3, 5]],), 4),
])

## Exercice 63 — Distance jusqu'à la sortie du labyrinthe

**Idée** : le **BFS** du thème 5 (`plus_court_chemin`), avec un départ et une arrivée marqués par des lettres au lieu d'être fixés aux coins. « Nombre minimal de déplacements » → BFS, jamais DFS.

**Complexité** : O(lignes × colonnes).

In [ ]:
from collections import deque

def distance_sortie(plan: list[str]) -> int:
    lignes, colonnes = len(plan), len(plan[0])
    for i in range(lignes):
        for j in range(colonnes):
            if plan[i][j] == "S":
                depart = (i, j)

    file = deque([(depart[0], depart[1], 0)])    # (ligne, colonne, nombre de pas)
    visites = {depart}
    while file:
        i, j, d = file.popleft()
        if plan[i][j] == "E":
            return d                             # première arrivée = plus court chemin (BFS)
        for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            ni, nj = i + di, j + dj
            if (0 <= ni < lignes and 0 <= nj < colonnes
                    and plan[ni][nj] != "#" and (ni, nj) not in visites):
                visites.add((ni, nj))
                file.append((ni, nj, d + 1))
    return -1


verifier(distance_sortie, [
    ((['S..', '##.', '..E'],), 4),
    ((['S#E'],), -1),
    ((['SE'],), 1),
    ((['S.#', '#..', '#.E'],), 4),
    ((['S...', '###.', 'E...'],), 8),
    ((['S.#', '..#', '##E'],), -1),
])

---

[Énoncés de l'examen](../examen_blanc_C.ipynb)